# Aula 03 — Probabilidade condicional e independência

## Objetivo

Calcular probabilidades condicionais em uma tabela, verificar produto e probabilidade total, testar independência e comparar por simulação distribuições conjuntas independentes e dependentes.

> Este notebook complementa a Aula 03 do módulo **Probabilidade, Estatística e Teoria da Informação**.

## Premissas e limites

- As frequências da tabela de mensagens descrevem uma população didática de 1.000 registros.
- A simulação segue o mecanismo declarado: prevalência de spam de 0,20, $P(L\mid S)=0,75$ e $P(L\mid S^c)=0,10$.
- A seed fixa ajuda a reproduzir a execução, mas não transforma estimativas em valores teóricos.
- A tolerância empírica é declarada antes dos testes.
- Associação probabilística não prova causalidade.
- A independência deve ser avaliada no contexto e não apenas assumida para simplificar o cálculo.

## 1. Preparação

O Google Colab já inclui NumPy e Matplotlib. Em ambiente local:

```bash
python -m pip install "numpy>=1.24" "matplotlib>=3.7" jupyter
```

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
TOL = 0.005
rng = np.random.default_rng(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seed:", SEED)
print("Tolerância empírica:", TOL)

## 2. Tabela de mensagens

As linhas representam a classe real e as colunas, a presença do padrão de link. O array contém contagens, não probabilidades.

| Classe real | Link | Sem link | Total |
|---|---:|---:|---:|
| Spam | 150 | 50 | 200 |
| Legítima | 80 | 720 | 800 |

In [ ]:
contagens = np.array([
    [150,  50],  # spam
    [ 80, 720],  # legítima
])

N = contagens.sum()
n_spam = contagens[0].sum()
n_link = contagens[:, 0].sum()
n_spam_link = contagens[0, 0]

p_spam = n_spam / N
p_link = n_link / N
p_spam_link = n_spam_link / N
p_spam_dado_link = n_spam_link / n_link
p_link_dado_spam = n_spam_link / n_spam

print(f"P(S)       = {p_spam:.4f}")
print(f"P(L)       = {p_link:.4f}")
print(f"P(S ∩ L)   = {p_spam_link:.4f}")
print(f"P(S | L)   = {p_spam_dado_link:.4f}")
print(f"P(L | S)   = {p_link_dado_spam:.4f}")

assert N == 1_000
assert np.isclose(p_spam, 0.20)
assert np.isclose(p_link, 0.23)
assert np.isclose(p_spam_dado_link, 150 / 230)
assert not np.isclose(p_spam_dado_link, p_link_dado_spam)

## 3. Função segura para a definição

A definição por razão requer $P(B)>0$ e $0\leq P(A\cap B)\leq P(B)\leq1$. A validação torna hipóteses inválidas visíveis.

In [ ]:
def prob_condicional(p_intersecao, p_condicao):
    if not 0 <= p_intersecao <= p_condicao <= 1:
        raise ValueError("Exija 0 ≤ P(A∩B) ≤ P(B) ≤ 1.")
    if p_condicao == 0:
        raise ZeroDivisionError("P(A|B) exige P(B) > 0 nesta definição.")
    return p_intersecao / p_condicao

calculada = prob_condicional(p_spam_link, p_link)
print(f"P(S | L) = {calculada:.6f}")
assert np.isclose(calculada, 150 / 230)

try:
    prob_condicional(0.0, 0.0)
except ZeroDivisionError as erro:
    print("Erro esperado:", erro)

## 4. Regra do produto e probabilidade total

Reconstruímos a conjunta por um caminho da árvore e a marginal do link pela soma dos caminhos disjuntos spam e legítima.

In [ ]:
p_link_dado_legitima = contagens[1, 0] / contagens[1].sum()
p_legitima = 1 - p_spam

conjunta_pelo_produto = p_spam * p_link_dado_spam
link_pela_total = (
    p_link_dado_spam * p_spam
    + p_link_dado_legitima * p_legitima
)

print(f"P(S ∩ L) pelo produto = {conjunta_pelo_produto:.4f}")
print(f"P(L) pela lei total   = {link_pela_total:.4f}")

assert np.isclose(conjunta_pelo_produto, p_spam_link)
assert np.isclose(link_pela_total, p_link)

## 5. Independência em um dado justo

Os eventos “par” e “múltiplo de 3” são independentes. “Par” e “ímpar” são disjuntos e, como ambos têm probabilidade positiva, são dependentes.

In [ ]:
omega = set(range(1, 7))
par = {2, 4, 6}
multiplo_3 = {3, 6}
impar = {1, 3, 5}

def p_uniforme(evento, espaco=omega):
    if not evento <= espaco or not espaco:
        raise ValueError("Evento e espaço amostral inválidos.")
    return len(evento) / len(espaco)

def relatorio_independencia(a, b, espaco=omega):
    p_a = p_uniforme(a, espaco)
    p_b = p_uniforme(b, espaco)
    p_ab = p_uniforme(a & b, espaco)
    produto = p_a * p_b
    return p_a, p_b, p_ab, produto, np.isclose(p_ab, produto)

r1 = relatorio_independencia(par, multiplo_3)
r2 = relatorio_independencia(par, impar)

print("par × múltiplo de 3:", r1)
print("par × ímpar:         ", r2)

assert r1[-1]
assert not r2[-1]
assert r2[2] == 0

## 6. Simulação do mecanismo das mensagens

Primeiro simulamos a classe. Depois, a probabilidade do link depende da classe: 0,75 para spam e 0,10 para legítima. Esse mecanismo produz dependência entre classe e link.

In [ ]:
N_SIMULACOES = 200_000

spam = rng.random(N_SIMULACOES) < 0.20
p_link_por_classe = np.where(spam, 0.75, 0.10)
link = rng.random(N_SIMULACOES) < p_link_por_classe

est_p_spam = spam.mean()
est_p_link = link.mean()
est_p_conjunta = (spam & link).mean()
est_p_spam_dado_link = (spam & link).sum() / link.sum()
est_p_link_dado_spam = (spam & link).sum() / spam.sum()

estimativas = {
    "P̂(S)": est_p_spam,
    "P̂(L)": est_p_link,
    "P̂(S ∩ L)": est_p_conjunta,
    "P̂(S | L)": est_p_spam_dado_link,
    "P̂(L | S)": est_p_link_dado_spam,
}

for nome, valor in estimativas.items():
    print(f"{nome:10} = {valor:.6f}")

assert abs(est_p_spam - p_spam) < TOL
assert abs(est_p_link - p_link) < TOL
assert abs(est_p_conjunta - p_spam_link) < TOL
assert abs(est_p_spam_dado_link - p_spam_dado_link) < TOL
assert abs(est_p_link_dado_spam - p_link_dado_spam) < TOL
print("Estimativas compatíveis com os valores teóricos na tolerância declarada.")

## 7. Mesmas marginais, dependência diferente

Nos dois cenários, cada bit vale 1 em aproximadamente metade dos casos. No primeiro, $X$ e $Y$ são gerados separadamente. No segundo, $Y$ copia $X$ em 80% dos casos, criando dependência sem mudar substancialmente as marginais.

In [ ]:
N_PARES = 200_000
x = rng.integers(0, 2, size=N_PARES)
y_independente = rng.integers(0, 2, size=N_PARES)
copia_x = rng.random(N_PARES) < 0.80
y_dependente = np.where(copia_x, x, 1 - x)

def tabela_conjunta(a, b):
    tabela = np.zeros((2, 2), dtype=float)
    np.add.at(tabela, (a, b), 1)
    return tabela / len(a)

t_ind = tabela_conjunta(x, y_independente)
t_dep = tabela_conjunta(x, y_dependente)

print("Conjunta independente:\n", np.round(t_ind, 4))
print("Conjunta dependente:\n", np.round(t_dep, 4))
print(f"Marginal P(X=1): {x.mean():.4f}")
print(f"Marginal P(Y_ind=1): {y_independente.mean():.4f}")
print(f"Marginal P(Y_dep=1): {y_dependente.mean():.4f}")

assert abs(t_ind[1, 1] - 0.25) < TOL
assert abs(t_dep[1, 1] - 0.40) < TOL
assert abs(x.mean() - 0.50) < TOL
assert abs(y_independente.mean() - 0.50) < TOL
assert abs(y_dependente.mean() - 0.50) < TOL

## 8. Visualização das distribuições conjuntas

Cada célula mostra uma probabilidade conjunta. Na matriz independente, as quatro células ficam próximas de 0,25. Na dependente, a massa se concentra na diagonal, embora as marginais permaneçam próximas de 0,50.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), constrained_layout=True)

for ax, tabela, titulo in zip(
    axes,
    [t_ind, t_dep],
    ["X e Y independentes", "Y copia X em 80% dos casos"],
):
    imagem = ax.imshow(tabela, vmin=0, vmax=0.5, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{tabela[i, j]:.3f}", ha="center", va="center",
                    color="white" if tabela[i, j] > 0.28 else "#102a43")
    ax.set_xticks([0, 1], labels=["Y=0", "Y=1"])
    ax.set_yticks([0, 1], labels=["X=0", "X=1"])
    ax.set_title(titulo)
    ax.set_xlabel("valor de Y")
    ax.set_ylabel("valor de X")

fig.colorbar(imagem, ax=axes, label="probabilidade conjunta", shrink=0.85)
plt.show()

## 9. Diagnóstico numérico de dependência

A diferença $P(A\cap B)-P(A)P(B)$ é zero sob independência exata. Em dados finitos, ela oscila ao redor de zero; um limiar descritivo não substitui um procedimento inferencial.

In [ ]:
def diagnostico_binario(a, b):
    p_a = np.mean(a)
    p_b = np.mean(b)
    p_ab = np.mean(a & b)
    esperado_ind = p_a * p_b
    return {
        "P(A)": p_a,
        "P(B)": p_b,
        "P(A∩B)": p_ab,
        "P(A)P(B)": esperado_ind,
        "diferença": p_ab - esperado_ind,
        "razão": p_ab / esperado_ind,
    }

diag_ind = diagnostico_binario(x == 1, y_independente == 1)
diag_dep = diagnostico_binario(x == 1, y_dependente == 1)

print("Cenário independente")
for chave, valor in diag_ind.items():
    print(f"  {chave:10}: {valor:.6f}")

print("Cenário dependente")
for chave, valor in diag_dep.items():
    print(f"  {chave:10}: {valor:.6f}")

assert abs(diag_ind["diferença"]) < TOL
assert diag_dep["diferença"] > 0.14

## 10. Desafios

1. Troque $P(L\mid S)$ de 0,75 para 0,40 e preveja $P(L)$ antes de executar.
2. Faça $P(L\mid S)=P(L\mid S^c)=0,10$. Classe e link ficam independentes no mecanismo simulado?
3. Aumente `N_SIMULACOES` e acompanhe a diferença entre estimativas e valores teóricos.
4. Mude a probabilidade de `copia_x` para 0,50. O segundo cenário fica independente?
5. Crie três bits $X$, $Y$ e $Z=X\oplus Y$ e verifique independência aos pares. Depois mostre que conhecer $X$ e $Y$ determina $Z$.
6. Modele dois serviços que compartilham uma falha de região. Compare a falha conjunta com o produto das marginais.
7. Escreva uma conclusão que diferencie associação, independência e causalidade para a tabela de mensagens.

## Conclusões

- Condicionar em $B$ restringe o universo e renormaliza sua massa.
- $P(A\mid B)$ e $P(B\mid A)$ compartilham a interseção, mas não o denominador.
- A regra geral é $P(A\cap B)=P(A)P(B\mid A)$.
- A probabilidade total soma caminhos disjuntos de uma partição.
- Independência requer fatoração; não é sinônimo de disjunção.
- Marginais semelhantes não garantem distribuições conjuntas semelhantes.
- Hipóteses de independência precisam de justificativa de domínio.

## Próxima etapa

Siga para a Aula 04 — **Teorema de Bayes e atualização de crenças**. Ela usará a probabilidade total para inverter condicionais e combinar prevalência com evidência.